In [ ]:
import json
import warnings
import random
from dataset import ImpactMeshDataset
from datamodule import ImpactMeshDataModule
from transforms import Compose, Normalize, RandomHorizontalFlip, RandomVerticalFlip
from utils import plot_sample, print_metadata

warnings.filterwarnings("ignore", message="Can't initialize NVML")

In [ ]:
# download a hazard release first: https://huggingface.co/datasets/paulhoehn/ImpactMesh-X
# config
ROOT = "/path/to/ImpactMesh-X/Flood"   # release folder for one hazard
TASK = "flood"                         # "flood" or "fire"
SPLIT_FILE = f"splits/{TASK}_splits.json"
NORM_FILE = f"splits/{TASK}_norm.json"

In [ ]:
# official train/val/test split -- positive (hazard-present) samples only, see README
with open(SPLIT_FILE) as f:
    split = json.load(f)
print(f"train={len(split['train'])}  val={len(split['val'])}  test={len(split['test'])}")

In [ ]:
# wrap in a DataModule, with the official per-band normalization stats
train_transforms = Compose([
    RandomHorizontalFlip(), RandomVerticalFlip(),
    Normalize(NORM_FILE),
])

dm = ImpactMeshDataModule(
    root=ROOT, task=TASK, pool="positive",
    train_ids=split["train"], val_ids=split["val"], test_ids=split["test"],
    train_transforms=train_transforms,
    batch_size=4, num_workers=2,
)
dm.setup()

In [ ]:
# look at one batch
batch = next(iter(dm.train_dataloader()))

print("batch keys:", list(batch.keys()))
for mod, x in batch["image"].items():
    print(f"  image[{mod}]: {tuple(x.shape)}  dtype={x.dtype}")
print(f"  mask: {tuple(batch['mask'].shape)}  dtype={batch['mask'].dtype}")
print(f"  metadata: list of {len(batch['metadata'])} per-sample dicts")

In [ ]:
# visualize a few samples (loading from the dataset, not the batch, since plot_sample wants one un-collated sample at a time)
# fixed indices for a few specific train-split events, swap for `random.sample(range(len(dm.train_dataset)), 3)` to get random ones instead
indices = [2822, 1369, 589] 

for i in indices:
    sample = dm.train_dataset[i]
    print_metadata(sample, task=TASK)
    plot_sample(sample, task=TASK)  # omit save_path to render inline

In [ ]:
# appendix: negative (hazard-free) samples -- not part of the official split, since that split is built for positive-sample hazard mapping only. 
# Load them as their own pool, build your own split over them if you need one 
ds_neg = ImpactMeshDataset(root=ROOT, task=TASK, pool="negative")
print(f"{len(ds_neg)} negative patches, no predefined split")

# fixed indices for a couple of specific negative-pool patches
neg_indices = [3889, 1443, 2145]

for i in neg_indices:
    sample = ds_neg[i]
    print_metadata(sample, task=TASK)
    plot_sample(sample, task=TASK)